## Import Libraries

In [1]:
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
import math 
from dateutil.relativedelta import relativedelta
from snowflake_utils import *
import os 
import shutil

In [2]:
from snowflake.snowpark.types import StringType
from darts import TimeSeries
from darts.dataprocessing.transformers import (
    Scaler,
    StaticCovariatesTransformer,
)
from darts.models import TFTModel

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Snowflake Session

In [3]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Python UDF

In [4]:
def remove_quotes_existing_columns(df):
    for old_col in df.columns:
        new_col = old_col.replace('"','')
        df = df.rename(old_col,new_col)
    return df

### Configuration

In [5]:
TABLE_NAME = 'MOP_DATABASE.SOQ.DAILY_FORECASTING_DATA_FOR_MODELLING_TFT_APR_23_TO_DEC_26'

### Read the table

In [6]:
data = session.table(TABLE_NAME)

In [7]:
data.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY").alias("UNIQUE_PARENT_DEALER_CODE_MODEL_FAMILIES")).show()

----------------------------------------------
|"UNIQUE_PARENT_DEALER_CODE_MODEL_FAMILIES"  |
----------------------------------------------
|117246                                      |
----------------------------------------------



### Segregate the training and test set

In [8]:
train_set = data.filter(F.col("CAL_DATE")<='2026-06-30')
test_set = data.filter(F.col("CAL_DATE")>='2026-07-01')

### Shape of training and test set

In [9]:
print(f"Shape of the training dataset is : {snowflake_utils.shape_of_snowpark_df(train_set)}")

print(f"Shape of the test dataset is : {snowflake_utils.shape_of_snowpark_df(test_set)}")

Shape of the training dataset is : (139171002, 86)
Shape of the test dataset is : (21573264, 86)


### Deleting the local folders

In [10]:
local_folders = ["./local_train_data",'./saved_group_keys']

for folder in local_folders:
    if os.path.exists(folder):
        try:
            shutil.rmtree(folder)
        except:
            print(f"Unable to delete {folder}")
        print(f"Successfully cleared out local folder: {folder}")

### Sanitize the snowpark dataframe

In [11]:
train_set_sanitized = train_set.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('<>'),F.lit('_')))

In [12]:
train_set_sanitized.select(F.col("PARENT_DEALER_CODE_MODEL_FAMILY")).distinct().show()

----------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"                 |
----------------------------------------------------
|11665_PLEASURE+_DRUM_SELF_CAST_RED                |
|11863_XTREME 160_DISC_SELF_CAST_GREY              |
|11038_GLAMOUR_DRUM_SELF_CAST_BLACK                |
|10861_SUPER SPLENDOR_DRUM_SELF_CAST_GREY          |
|11840_PLEASURE+_DRUM_SELF_SHEET METAL_BLACK       |
|10285_XTREME 125_DISC_SELF_CAST_ORANGE            |
|10051_SUPER SPLENDOR_DRUM_SELF_CAST_SPORTS RED    |
|11505_SUPER SPLENDOR_DRUM_SELF_CAST_BLACK         |
|11961_SPLENDOR+_DRUM_SELF_CASTI_BLACK RED PURPLE  |
|10721_SPLENDOR+_DRUM_SELF_CAST_RED BLACK          |
----------------------------------------------------



In [13]:
test_set_sanitized = test_set.with_column("PARENT_DEALER_CODE_MODEL_FAMILY",F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"),F.lit('<>'),F.lit('_')))

### Saving the partitioned files in snowflake stage

In [14]:
# train_stage_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/train_daily_forecast_parquet_files"
# test_stage_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/test_daily_forecast_parquet_files"
stage_train_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/train_daily_forecast_sanitized"
stage_test_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/test_daily_forecast_sanitized"



In [15]:
train_set_sanitized.write.parquet(
    stage_train_path, 
    partition_by="PARENT_DEALER_CODE_MODEL_FAMILY"
)

[Row(rows_unloaded=139171002, input_bytes=3556298842, output_bytes=3556298842)]

In [16]:
test_set_sanitized.write.parquet(
    stage_test_path, 
    partition_by="PARENT_DEALER_CODE_MODEL_FAMILY"
)

[Row(rows_unloaded=21573264, input_bytes=2669808666, output_bytes=2669808666)]

### Saving the group_keys

In [17]:
group_keys = [row["PARENT_DEALER_CODE_MODEL_FAMILY"] for row in train_set_sanitized.select("PARENT_DEALER_CODE_MODEL_FAMILY").distinct().collect()]

In [18]:
import os 
import shutil
import json
folder_name = 'saved_group_keys'

# if os.path.exists(folder_name):
#     shutil.rmtree(folder_name)

os.makedirs(folder_name,exist_ok=True)

In [19]:
local_json_path = os.path.join(folder_name, "group_keys.json")
with open(local_json_path, "w") as f:
    json.dump(group_keys, f)

### Saving the variables type

In [20]:
time_col = 'CAL_DATE'
group_col = 'PARENT_DEALER_CODE_MODEL_FAMILY'
target_col = 'NET_SALES'

id_cols = ['CAL_DATE','PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']

In [21]:
static_covariates = [field.name for field in data.schema if isinstance(field.datatype,StringType) and field.name not in ['MODEL_FAMILY_CODE','DAY_OF_THE_WEEK','PARENT_DEALER_CODE_MODEL_FAMILY']]

future_covariates = [col for col in data.columns if col not in static_covariates and col not in id_cols and col not in ['MODEL_FAMILY_CODE','DAY_OF_THE_WEEK','YEAR','MONTH','DAY_OF_THE_MONTH']]

In [22]:
static_covariates

['PARENT_DEALER_CODE',
 'MODEL_FAMILY',
 'MODEL_NAME',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME']

In [23]:
future_covariates

['NEW_YEAR',
 'LOHRI',
 'MAKAR_SANKRANTI',
 'REPUBLIC_DAY',
 'VASANT_PANCHAMI',
 'MAHA_SHIVRATRI',
 'EID_UL_FITR',
 'HOLIKA_DAHAN',
 'HOLI',
 'HANUMAN_JAYANTI',
 'AKSHAYA_TRITYA',
 'BUDDHA_PURNIMA',
 'GANGA_DUSSEHRA',
 'JAGANNATH_RATHYATRA',
 'GURU_PURNIMA',
 'NAG_PANCHAMI',
 'RAKSHA_BANDHAN',
 'HARTALIK_TEEJ',
 'GANESH_CHATURTHI',
 'JANMASHTAMI',
 'VISHWAKARMA_PUJA',
 'KARWA_CHAUTH',
 'ONAM',
 'MARRIAGE_DAY',
 '"N-16"',
 '"N-15"',
 '"N-14"',
 '"N-13"',
 '"N-12"',
 '"N-11"',
 '"N-10"',
 '"N-9"',
 '"N-8"',
 '"N-7"',
 '"N-6"',
 '"N-5"',
 '"N-4"',
 '"N-3"',
 '"N-2"',
 '"N-1"',
 'N',
 '"N+1"',
 '"N+2"',
 '"N+3"',
 '"N+4"',
 '"N+5"',
 '"N+6"',
 '"N+7"',
 '"N+8"',
 '"N+9"',
 '"N+10"',
 '"D-3"',
 '"D-2"',
 '"D-1"',
 'D',
 '"D+1"',
 '"D+2"',
 '"D+3"',
 '"D+4"',
 '"D+5"',
 '"D+6"',
 'C',
 '"C+1"',
 '"C+2"',
 '"C+3"',
 '"C+4"',
 '"C+5"',
 '"C+6"']

### Add encoders

In [24]:
add_encoders={
    'cyclic': {'future': ['month','dayofweek','dayofyear']},
    'transformer': Scaler()
}

### Downloading files from Snowflake Stage

In [25]:
stage_base = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES"
stage_train_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/train_daily_forecast_sanitized"
stage_test_path = "@MOP_DATABASE.SOQ.TFT_DAILY_FORECASTING_FILES/test_daily_forecast_sanitized"
local_train_dir = "./local_train_data"
local_test_dir = "./local_test_data"

In [26]:
# 2. Download the training partitioned data
print("Downloading partitioned TRAINING files to local storage...")
session.file.get(stage_train_path, local_train_dir)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: './local_train_data\\data_01c5c47c-0001-dd74-01a2-ce012449884b_009_5_0.snappy.parquet'

In [ ]:
print("Downloading partitioned TEST files to local storage...")
session.file.get(stage_test_path,local_test_dir)

In [ ]:
stage_train_path

In [ ]:
absolute_local_path = os.path.abspath(local_train_dir).replace("\\", "/")

print("Resuming download using controlled parallelism...")

# Using raw SQL to leverage OVERWRITE=FALSE (resuming progress)
# and lowering PARALLEL to stabilize Windows I/O
sql_command = f"""
    GET {stage_train_path} 
    file://{absolute_local_path} 
    PARALLEL = 2 
    OVERWRITE = FALSE;
"""

try:
    results = session.sql(sql_command).collect()
    print(f"Download resumed successfully! Processed {len(results)} remaining files.")
except Exception as e:
    print(f"Download execution error: {e}")

session.close()

Resuming download using controlled parallelism...
Download execution error: 253002: While getting file(s) there was an error: 'PermissionError(13, 'Permission denied')', this might be caused by your access to the blob storage provider, or by Snowflake.
